# Virasoro 代数的 $C_2$ 代数计算演示

本 notebook 演示如何使用 pyope 对 **Virasoro 代数** 进行 $C_2$ 商空间分析，以及 null 态搜索。

## 理论背景

给定顶点算符代数 $(V, Y, |0\rangle, T)$，定义 $C_2$ 子空间：
$$C_2(V) = \operatorname{span}\{a_{(-2)}b : a, b \in V\} = \operatorname{span}\{:(\partial a) \phi:\}$$

**Zhu $C_2$-cofiniteness 条件**：若 $\dim(V / C_2(V)) < \infty$，则 $V$ 是 $C_2$-cofinite 的。

**商空间中的幂零性**：在 $R_V = V/C_2(V)$ 中，若 $[T^n] = 0$，则 $T$ 在 $R_V$ 中是幂零的。对于有限中心荷 $c$，这恰好对应着 Virasoro Verma 模中存在奇异态 $L_{-1}^n|h\rangle \sim 0$（在适当条件下）。

## 计算路径

1. $C_2$ 子空间的生成元 $\{:(\partial T)\tilde{\phi}:\}$
2. 商空间约化：`GenericC2Reducer.quotient_normal_form(expr)` 
3. Null 态搜索：`quotient_precheck` → `search_from_sources`

In [3]:
from pyope import *
from sympy import Symbol

print("pyope 导入成功")

pyope 导入成功


## 第 1 部分：定义 Virasoro 代数

Virasoro 代数只有一个生成元：能动张量 $T$（conformal weight = 2），OPE 为：
$$T(z)T(w) \sim \frac{c/2}{(z-w)^4} + \frac{2T(w)}{(z-w)^2} + \frac{\partial T(w)}{z-w}$$

In [16]:
# 定义应力张量，conformal weight = 2
clear_registry()
T = BasisOperator("T", conformal_weight=2)
Bosonic(T)  # T 是玻色子

# 中心荷（符号参数）
c = -22/5

# 注册 Virasoro OPE：T·T 的奇点 (c/2, 0, 2T, ∂T)
OPE[T, T] = MakeOPE([c / 2 * One, 0, 2 * T, d(T)])

print("Virasoro 代数定义完成")
print(f"T 的 conformal weight = {T.conformal_weight}")

Virasoro 代数定义完成
T 的 conformal weight = 2


In [17]:
# 构建算符基底，最大权重设为 8
basis = LocalOperatorBasis([T], max_weight=8)

# 列出各权重的基底元素
for w in range(1, 9):
    b = basis.list(w)
    if b:
        print(f"权重 {w}: {b}")

权重 2: [BasisOperator('T')]
权重 3: [d(T)]
权重 4: [d^2(T), NO(T, T)]
权重 5: [d^3(T), NO(∂T, T)]
权重 6: [d^4(T), NO(T, NO(T,T)), NO(∂T, ∂T), NO(∂^2T, T)]
权重 7: [d^5(T), NO(∂T, NO(T,T)), NO(∂^2T, ∂T), NO(∂^3T, T)]
权重 8: [d^6(T), NO(T, NO(T,NO(T,T))), NO(∂T, NO(∂T,T)), NO(∂^2T, NO(T,T)), NO(∂^2T, ∂^2T), NO(∂^3T, ∂T), NO(∂^4T, T)]


## 第 2 部分：$C_2$ 子空间

$C_2$ 子空间在权重 $h$ 处由下面的生成元张成：
$$C_2(V)_h = \operatorname{span}\{:(\partial a)\phi: \mid a \in V_{h_a},\ \phi \in V_{h-h_a-1}\}$$

对 Virasoro 代数，唯一的基本生成元是 $T$（权重 2），因此 $C_2$ 生成元形如 $:(\partial T)\phi:$。

In [18]:
c2 = C2Space(basis)

print("=== C2 子空间生成元 ===")
for w in range(3, 9):
    gens = c2.generators(w)
    print(f"权重 {w}: {gens if gens else '（空）'}")

=== C2 子空间生成元 ===
权重 3: [d(T)]
权重 4: [d^2(T)]
权重 5: [∂^3T/6 + 2*NO(∂T,T), d^3(T), NO(∂T, T)]
权重 6: [NO(∂T,∂T) + NO(∂^2T,T), d^4(T), NO(∂T, ∂T), NO(∂^2T, T)]
权重 7: [-11*∂^5T/300 + 2*NO(∂T,NO(T,T)) + NO(∂^2T,∂T) + 5*NO(∂^3T,T)/6, ∂^5T/40 + 3*NO(∂T,NO(T,T)) + NO(∂^3T,T)/2, ∂^5T/60 + 2*NO(∂^2T,∂T), ∂^5T/60 + NO(∂^2T,∂T), NO(∂^2T,∂T) + NO(∂^3T,T), d^5(T), NO(∂T, NO(T,T)), NO(∂^2T, ∂T), NO(∂^3T, T)]
权重 8: [-11*∂^6T/1800 + NO(∂T,NO(∂T,T)) + NO(∂^2T,NO(T,T)) + NO(∂^3T,∂T)/3 + NO(∂^4T,T)/6, -17*∂^6T/900 + 2*NO(∂T,NO(∂T,T)) + NO(∂^2T,∂^2T) + 3*NO(∂^3T,∂T)/2 + NO(∂^4T,T)/3, ∂^6T/360 + 2*NO(∂T,NO(∂T,T)) + NO(∂^2T,NO(T,T)) + NO(∂^3T,∂T)/6, ∂^6T/60 + NO(∂^3T,∂T), NO(∂^2T,∂^2T) + NO(∂^3T,∂T), NO(∂^3T,∂T) + NO(∂^4T,T), d^6(T), NO(∂T, NO(∂T,T)), NO(∂^2T, NO(T,T)), NO(∂^2T, ∂^2T), NO(∂^3T, ∂T), NO(∂^4T, T)]


In [19]:
print("=== C2 子空间线性无关基底 ===")
for w in range(3, 9):
    b = c2.basis(w)
    full_basis = basis.list(w)
    print(f"权重 {w}: C2 基 = {b}, 全基维数 = {len(full_basis)}, C2 维数 = {len(b)}")

=== C2 子空间线性无关基底 ===
权重 3: C2 基 = [d(T)], 全基维数 = 1, C2 维数 = 1
权重 4: C2 基 = [d^2(T)], 全基维数 = 2, C2 维数 = 1
权重 5: C2 基 = [∂^3T/6 + 2*NO(∂T,T), d^3(T)], 全基维数 = 2, C2 维数 = 2
权重 6: C2 基 = [NO(∂T,∂T) + NO(∂^2T,T), d^4(T), NO(∂T, ∂T)], 全基维数 = 4, C2 维数 = 3
权重 7: C2 基 = [-11*∂^5T/300 + 2*NO(∂T,NO(T,T)) + NO(∂^2T,∂T) + 5*NO(∂^3T,T)/6, ∂^5T/40 + 3*NO(∂T,NO(T,T)) + NO(∂^3T,T)/2, ∂^5T/60 + 2*NO(∂^2T,∂T), ∂^5T/60 + NO(∂^2T,∂T)], 全基维数 = 4, C2 维数 = 4
权重 8: C2 基 = [-11*∂^6T/1800 + NO(∂T,NO(∂T,T)) + NO(∂^2T,NO(T,T)) + NO(∂^3T,∂T)/3 + NO(∂^4T,T)/6, -17*∂^6T/900 + 2*NO(∂T,NO(∂T,T)) + NO(∂^2T,∂^2T) + 3*NO(∂^3T,∂T)/2 + NO(∂^4T,T)/3, ∂^6T/360 + 2*NO(∂T,NO(∂T,T)) + NO(∂^2T,NO(T,T)) + NO(∂^3T,∂T)/6, ∂^6T/60 + NO(∂^3T,∂T), NO(∂^2T,∂^2T) + NO(∂^3T,∂T), NO(∂^3T,∂T) + NO(∂^4T,T)], 全基维数 = 7, C2 维数 = 6


In [20]:
# 检验具体元素是否属于 C2
print("=== C2 成员判定 ===")
print(f":∂T · T: ∈ C2? → {c2.contains(NO(d(T), T))}   ← 期望 True")
print(f":TT: ∈ C2?      → {c2.contains(NO(T, T))}   ← 期望 False")
print(f"∂T ∈ C2?        → {c2.contains(d(T))}   ← 期望 True（:(∂T)·1: = ∂T ∈ C2）")
print(f":∂T·∂T: ∈ C2?  → {c2.contains(NO(d(T), d(T)))}   ← 期望 True（a=T, φ=∂T → :(∂T)(∂T): ∈ C2）")

=== C2 成员判定 ===
:∂T · T: ∈ C2? → True   ← 期望 True
:TT: ∈ C2?      → False   ← 期望 False
∂T ∈ C2?        → True   ← 期望 True（:(∂T)·1: = ∂T ∈ C2）
:∂T·∂T: ∈ C2?  → True   ← 期望 True（a=T, φ=∂T → :(∂T)(∂T): ∈ C2）


## 第 3 部分：稀疏 $C_2$ 约化器（`GenericC2Reducer`）

`GenericC2Reducer` 对给定算符 $\phi$ 计算其在商空间 $R_V = V/C_2(V)$ 中的**正规形式**（quotient normal form），以及**分解 witness**（decomposition witness）：

$$\phi = \underbrace{\phi_{C_2}}_{\in C_2(V)} + \underbrace{\phi_{\text{rem}}}_{\text{quotient remainder}}$$

In [21]:
reducer = GenericC2Reducer(basis)

print("=== 商空间正规形式 ===")
test_exprs = [
    (NO(d(T), T),     "0",       "期望 0（C2 元素）"),
    (NO(T, T),        ":TT:",           "期望 :TT:（非 C2）"),
    (d(T),            "0",             "期望 0（C2 元素：:(∂T)·1: = ∂T）"),
    (NO(d(T), d(T)),  "0",    "期望 0（C2 元素：a=T, φ=∂T）"),
]

for expr, label, note in test_exprs:
    nf = reducer.quotient_normal_form(expr)
    print(f"  {label:20s} → {nf}  ({note})")

=== 商空间正规形式 ===
  0                    → Zero  (期望 0（C2 元素）)
  :TT:                 → NO(T,T)  (期望 :TT:（非 C2）)
  0                    → Zero  (期望 0（C2 元素：:(∂T)·1: = ∂T）)
  0                    → Zero  (期望 0（C2 元素：a=T, φ=∂T）)


In [22]:
# 查看完整的 C2 分解 witness
witness = reducer.solve_c2_witness(NO(d(T), T))

print("=== C2ReductionWitness for :(∂T)T: ===")
print(f"  表达式 expr        = {witness.expr}")
print(f"  商余式 remainder   = {witness.remainder}   ← 0 表示属于 C2")
print(f"  C2 分量 c2_part    = {witness.c2_part}")
print(f"  C2 生成元 generators = {witness.generators}")
print(f"  分解系数 coefficients = {witness.coefficients}")
print(f"  目标权重            = {witness.target_weight}")

=== C2ReductionWitness for :(∂T)T: ===
  表达式 expr        = NO(∂T,T)
  商余式 remainder   = Zero   ← 0 表示属于 C2
  C2 分量 c2_part    = NO(∂T,T)
  C2 生成元 generators = [∂^3T/6 + 2*NO(∂T,T), d^3(T)]
  分解系数 coefficients = [1/2, -1/12]
  目标权重            = 5


In [23]:
# 对权重 8 的算符 :TTT: 做 C2 约化
ttt = basis.canonicalize(NO(T, NO(T, T)))
witness_ttt = reducer.solve_c2_witness(ttt)

print("=== C2ReductionWitness for :TTT: (weight 6) ===")
print(f"  商余式 remainder   = {witness_ttt.remainder}")
print(f"  C2 生成元          = {witness_ttt.generators}")
print(f"  分解系数           = {witness_ttt.coefficients}")
print()
print("验证：c2_part + remainder = expr?")
diff = basis.canonicalize(witness_ttt.c2_part + witness_ttt.remainder - ttt)
print(f"  误差 = {diff}")

=== C2ReductionWitness for :TTT: (weight 6) ===
  商余式 remainder   = NO(T,NO(T,T))
  C2 生成元          = [NO(∂T,∂T) + NO(∂^2T,T), d^4(T), NO(∂T, ∂T)]
  分解系数           = [0, 0, 0]

验证：c2_part + remainder = expr?
  误差 = Zero


## 第 4 部分：应力张量的商空间幂零性

在 $R_V = V/C_2(V)$ 中，$[T]$ 满足什么幂零关系？

我们检验 $[T^n] = 0$（即 $T^n \in C_2(V)$）对哪个 $n$ 成立。根据 Zhu 的理论，对 Virasoro 代数，$R_V \cong \mathbb{C}[T]/(T^2) \cdot \{\text{...}\}$ 的结构取决于中心荷。运算上，我们直接用 `is_zero_mod_c2` 检验。

In [50]:
OPE(T, simplify(NO(T, T) - 3/10 * d(T, 2))).pole(2) - 4 * (NO(T, T) - 3/10 * d(T, 2))

simplify(OPE(T, simplify(NO(T, T) - 3/10 * d(T, 2))).pole(1) - d(NO(T, T) - 3/10 * d(T, 2)))

0

In [24]:
print("=== T 在商空间中的幂次 ===")
print("检验 [T^n] = 0 in R_V = V/C2(V):\n")

for n in range(1, 6):
    T_power = basis.canonicalize(NO(*([T] * n)))
    is_zero = reducer.is_zero_mod_c2(T_power)
    nf = reducer.quotient_normal_form(T_power)
    print(f"  n={n}: T^{n} = {T_power}")
    print(f"        商余式 = {nf}")
    print(f"        在 C2 中? {is_zero}")
    print()

=== T 在商空间中的幂次 ===
检验 [T^n] = 0 in R_V = V/C2(V):

  n=1: T^1 = T
        商余式 = T
        在 C2 中? False

  n=2: T^2 = NO(T,T)
        商余式 = NO(T,T)
        在 C2 中? False

  n=3: T^3 = NO(T,NO(T,T))
        商余式 = NO(T,NO(T,T))
        在 C2 中? False

  n=4: T^4 = NO(T,NO(T,NO(T,T)))
        商余式 = NO(T,NO(T,NO(T,T)))
        在 C2 中? False

  n=5: T^5 = NO(T,NO(T,NO(T,NO(T,T))))
        商余式 = NO(T,NO(T,NO(T,NO(T,T))))
        在 C2 中? False



## 第 5 部分：Null 态搜索（`C2NullSearcher`）

`C2NullSearcher` 实现两阶段搜索：

### 阶段 1：商空间预检（`quotient_precheck`）
检验目标 $\phi$ 在 $R_V$ 中是否为零。若**非零（obstructed）**，则 $\phi$ 绝不是 null 态（无需进一步搜索）。

### 阶段 2：后代空间提升（`search_from_sources`）
在商空间中，从给定来源算符出发生成后代空间，寻找满足 $[\chi] = [\phi]$ 的后代 $\chi$，即：
$$\chi - \phi \in C_2(V)$$
同时验证 $\chi$ 满足奇异向量条件 $L_n \chi = 0$（$n > 0$）。

In [ ]:
# 使用 C2NullSearcher（高层次 API）
searcher = C2NullSearcher(basis, stress_tensor=T)

print("=== 商空间预检 ===")

# 预检 :(∂T)T:，应该 needs_lift（余式为 0）
precheck_c2 = searcher.quotient_precheck(NO(d(T), T))
print(f":(∂T)T: → status = '{precheck_c2.status}'")
print(f"          商余式 = {precheck_c2.quotient_remainder}")
print()

# 预检 :TT:，应该 obstructed（余式非零，不可能是 null）
precheck_tt = searcher.quotient_precheck(NO(T, T))
print(f":TT: → status = '{precheck_tt.status}'")
print(f"       商余式 = {precheck_tt.quotient_remainder}")

In [ ]:
# 搜索 T 的幂次作为 null 态（使用后代提升）
print("=== 应力张量幂零性搜索 ===")
print("（sources=[T] 表示从 T 生成后代空间）\n")

for n in range(1, 5):
    result = searcher.search_stress_tensor_nilpotency(n, sources=[T])
    if result is None:
        print(f"n={n}: 未找到解")
    else:
        status = result.get('status', result.status if hasattr(result, 'status') else '?')
        print(f"n={n}: status = '{result['status']}'")
        if result['status'] == 'solved':
            print(f"      null_operator = {result['null_operator']}")
            print(f"      c2_remainder  = {result['c2_remainder']}")
        else:
            print(f"      obstruction   = {result.get('obstruction', '?')}")
        print()

In [ ]:
# 搜索 :TT: 的后代分解
print("=== 搜索 :TT:（权重 4）的后代分解 ===")
result_tt = searcher.search_from_sources(4, [T], NO(T, T))

if result_tt is not None:
    print(f"status        = {result_tt['status']}")
    print(f"null_operator = {result_tt['null_operator']}")
    print(f"c2_remainder  = {result_tt['c2_remainder']}")
    print(f"descendant_basis  = {result_tt['descendant_basis']}")
    print(f"descendant_coeffs = {result_tt['descendant_coeffs']}")
else:
    print(":TT: 的后代分解不存在（obstructed 或无解）")

## 第 6 部分：$C_2$ 商空间维数（Hilbert 级数）

我们计算各权重处 $R_V = V/C_2(V)$ 的维数：
$$\dim R_V[h] = \dim V[h] - \dim C_2(V)[h]$$

对 Virasoro 代数，$V[h]$ 是 $h$ 处的 Verma 模基底数（分拆数），而 $C_2(V)[h]$ 是 $C_2$ 子空间的维数。

In [ ]:
print("=== Virasoro 代数商空间维数 ===")
print(f"{'权重 h':>10} | {'dim V[h]':>10} | {'dim C2[h]':>10} | {'dim R_V[h]':>12}")
print("-" * 50)

for w in range(0, 10):
    full_dim = len(basis.list(w)) if w >= 2 else (1 if w == 0 else 0)
    # 对权重 0，只有 |0⟩；权重 1 没有基元素（T 是权重 2 的生成元）
    if w == 0:
        full_b = []
        full_dim = 0  # 真空态不计入算符基
    else:
        full_b = basis.list(w)
        full_dim = len(full_b)
    
    c2_b = c2.basis(w) if w >= 3 else []
    c2_dim = len(c2_b)
    quotient_dim = full_dim - c2_dim
    
    print(f"  h = {w:3d}   | {full_dim:10d} | {c2_dim:10d} | {quotient_dim:12d}")

## 小结

| 计算任务 | 使用的类/函数 |
|---------|------------|
| 构造 $C_2$ 子空间生成元和基底 | `C2Space.generators()`, `C2Space.basis()` |
| 判断算符是否属于 $C_2$ | `C2Space.contains()` 或 `GenericC2Reducer.is_zero_mod_c2()` |
| 获取商空间正规形式 | `GenericC2Reducer.quotient_normal_form()` |
| 求 $C_2$ 分解的完整 witness | `GenericC2Reducer.solve_c2_witness()` |
| Null 态搜索（商空间预检） | `C2NullSearcher.quotient_precheck()` |
| Null 态搜索（后代提升） | `C2NullSearcher.search_from_sources()` |
| 应力张量幂零性 | `C2NullSearcher.search_stress_tensor_nilpotency()` |

**$C_2$ 代数的核心关系**：
$$\phi \in C_2(V) \iff [\phi] = 0 \text{ in } R_V = V/C_2(V)$$
$$\phi \text{ 是 null 态} \implies [\phi] = 0 \text{ in } R_V$$